#  Tugas Besar Big Data: ELT Pipeline (NYC 311 & Weather)
Kelompok:
- Pricilia Apriana (101032300056) -ETL
- Atha Aulia Shidiq (101032300197) - ELT
---------------------------------------------
**Arsitektur:** ELT (Extract, Load, Transform) menggunakan PostgreSQL (Neon DB)

Notebook ini menjalankan keseluruhan *pipeline* data. Berbeda dengan ETL yang melakukan transformasi di memori menggunakan Pandas, pendekatan ELT ini memuat data mentah secara langsung ke *Data Warehouse*, lalu mengeksekusi seluruh pembersihan dan *feature engineering* secara *native* di dalam *database* menggunakan SQL.

In [14]:
!pip install -q pg8000

In [15]:
import ssl
import getpass
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# Pisah komponen - hindari karakter spesial di URL
USER = "neondb_owner"

#Pass Neon DB
print("Membangun koneksi ke Neon DB...")
raw_password = getpass.getpass("Masukkan Password Neon DB Anda: ")
PASSWORD = quote_plus(raw_password)

HOST = "ep-calm-bar-ao8ujkdt-pooler.c-2.ap-southeast-1.aws.neon.tech"
DB = "neondb"

# 1. Hapus '?ssl=require' dari akhir URL
DATABASE_URL = f"postgresql+pg8000://{USER}:{PASSWORD}@{HOST}/{DB}"

# 2. Buat SSL context standar dari bawaan Python
ssl_context = ssl.create_default_context()

# 3. Masukkan ssl_context ke dalam argument koneksi (connect_args)
engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,
    pool_recycle=300,
    connect_args={'ssl_context': ssl_context}
)

with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print("Connected to:", result.fetchone()[0])

Membangun koneksi ke Neon DB...
Masukkan Password Neon DB Anda: ··········
Connected to: PostgreSQL 18.4 (6e15e70) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


## PHASE 1: Setup & Extract
Menghubungkan Colab dengan Google Drive untuk mengakses sumber data mentah (Raw Data), yaitu file CSV NYC 311 (500k baris) dan file JSON historis cuaca dari Open-Meteo.

In [16]:
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/bigdata_uas_2026'
RAW_PATH = f'{BASE_PATH}/raw'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import os
# Cek file raw data
print(os.listdir(RAW_PATH))
# Output harusnya: ['nyc311_raw.csv', 'weather_raw.json']

['nyc311_raw.csv', 'weather_raw.json']


In [18]:
import pandas as pd
import time

# Cek jumlah baris yang sudah aman di database
with engine.connect() as conn:
    try:
        result = conn.execute(text("SELECT COUNT(*) FROM elt_raw_311;"))
        row_count = result.fetchone()[0]
        print(f"Data yang sudah masuk: {row_count:,} baris.")
    except Exception as e:
        print("Tabel belum ada atau kosong. Jumlah baris: 0")
        row_count = 0

Data yang sudah masuk: 500,000 baris.


## PHASE 2: Load Data (Staging)
Memuat data mentah ke dalam *Data Warehouse* (Neon DB).
Untuk menghindari *memory overload* dan *timeout* dari server, data dimuat menggunakan teknik **Chunking** dan dioptimasi menggunakan fungsi `psql_insert_copy` (metode `COPY` PostgreSQL yang jauh lebih cepat daripada `INSERT` standar). Semua data dimuat sebagai teks murni (`dtype=str`) untuk menjaga keaslian raw data.

In [19]:
import pandas as pd
import time
import csv
from io import StringIO
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# ==============================================================================
# 1. SETUP KONEKSI TANGGUH (Menggunakan psycopg2 bawaan standar)
# ==============================================================================
BASE_PATH = '/content/drive/MyDrive/bigdata_uas_2026'
RAW_PATH = f'{BASE_PATH}/raw'

USER = "neondb_owner"
PASSWORD = quote_plus(raw_password)
HOST = "ep-calm-bar-ao8ujkdt-pooler.c-2.ap-southeast-1.aws.neon.tech"
DB = "neondb"

# Menggunakan format psycopg2 standar
DATABASE_URL = f"postgresql://{USER}:{PASSWORD}@{HOST}/{DB}?sslmode=require"
engine = create_engine(DATABASE_URL)

# ==============================================================================
# 2. FUNGSI OPTIMASI INSERT
# ==============================================================================
def psql_insert_copy(table, conn, keys, data_iter):
    """Mengeksekusi pengiriman data batch di dalam chunk secara efisien"""
    dbapi_conn = conn.connection
    with dbapi_conn.cursor() as cur:
        s_buf = StringIO()
        writer = csv.writer(s_buf)
        writer.writerows(data_iter)
        s_buf.seek(0)
        columns = ', '.join('"{}"'.format(k) for k in keys)
        sql = f'COPY {table.name} ({columns}) FROM STDIN WITH CSV'
        cur.copy_expert(sql=sql, file=s_buf)

# ==============================================================================
# 3. EKSEKUSI LOAD DATA (Anti-Gagal Tipe Data)
# ==============================================================================
table_311 = 'elt_raw_311'
file_311 = f"{RAW_PATH}/nyc311_raw.csv"

# Menggunakan chunk 5000 sesuai pedoman project
CHUNK_SIZE = 5000
MAX_ROWS = 500000
total_loaded = 0

print(f"[STAGE] Mulai proses Extract & Load dengan Chunking {CHUNK_SIZE} baris...")
start_time = time.time()

try:
    # TAMBAHAN dtype=str: Memaksa semua data dibaca sebagai teks/string murni
    # Ini menjamin tidak ada konflik tipe data saat insert ke PostgreSQL (konsep murni ELT)
    with pd.read_csv(file_311, chunksize=CHUNK_SIZE, low_memory=False, dtype=str) as reader:
        for i, chunk in enumerate(reader):
            if_exists_mode = 'replace' if i == 0 else 'append'

            # Load chunk ke database
            chunk.to_sql(
                name=table_311,
                con=engine,
                if_exists=if_exists_mode,
                index=False,
                method=psql_insert_copy # Menggunakan optimasi agar server tidak putus
            )

            total_loaded += len(chunk)
            print(f"  -> Berhasil load {total_loaded:,} baris...")

            if total_loaded >= MAX_ROWS:
              print(f"\n  -> Yay! Mencapai batas limit sampel {MAX_ROWS:,} baris.")
              break

    print(f"[STAGE] Selesai load {table_311} dalam {time.time() - start_time:.2f} detik.\n")

except Exception as e:
    print(f"\n[ERROR] Terjadi kegagalan saat load {table_311}: {e}")

[STAGE] Mulai proses Extract & Load dengan Chunking 5000 baris...

[ERROR] Terjadi kegagalan saat load elt_raw_311: (psycopg2.errors.DependentObjectsStillExist) cannot drop table elt_raw_311 because other objects depend on it
DETAIL:  view elt_clean_311 depends on table elt_raw_311
view elt_std_311 depends on view elt_clean_311
view elt_fact_311_analytics depends on view elt_std_311
HINT:  Use DROP ... CASCADE to drop the dependent objects too.

[SQL: 
DROP TABLE elt_raw_311]
(Background on this error at: https://sqlalche.me/e/20/2j85)


In [20]:
import pandas as pd
import time

# Cek jumlah baris yang sudah aman di database
with engine.connect() as conn:
    try:
        result = conn.execute(text("SELECT COUNT(*) FROM elt_raw_311;"))
        row_count = result.fetchone()[0]
        print(f"Data yang sudah masuk: {row_count:,} baris.")
    except Exception as e:
        print("Tabel belum ada atau kosong. Jumlah baris: 0")
        row_count = 0

Data yang sudah masuk: 500,000 baris.


Memuat data cuaca. Sesuai prinsip ELT murni, JSON tidak di-*parsing* di Python. Seluruh struktur JSON dimasukkan utuh ke dalam satu baris kolom tipe `TEXT` untuk nantinya dibongkar oleh mesin PostgreSQL.

In [21]:
import pandas as pd
import json
import time

print("[STAGE] Memuat Data Weather (100% Pure ELT - Raw JSON)...")
start_time_weather = time.time()

file_weather = f"{RAW_PATH}/weather_raw.json"
table_weather = 'elt_raw_weather'

try:
    # 1. Buka dan baca file JSON mentah
    with open(file_weather, 'r') as f:
        weather_data = json.load(f)

    # 2. ELT MURNI: Jangan ekstrak apa pun di Python!
    # Kita bungkus seluruh isi JSON utuh ke dalam satu kolom teks bernama 'raw_json_data'
    df_weather = pd.DataFrame({'raw_json_data': [json.dumps(weather_data)]})

    # 3. Lempar 1 baris utuh tersebut ke Neon
    df_weather.to_sql(
        name=table_weather,
        con=engine,
        if_exists='replace',
        index=False
    )

    print(f"  -> Berhasil load {len(df_weather)} baris (Berisi bongkahan utuh Raw JSON) ke '{table_weather}'.")
    print(f"[STAGE] Selesai memuat {table_weather} dalam {time.time() - start_time_weather:.2f} detik.")
    print("\n💡 Catatan ELT: Data sengaja dimuat sebagai 1 baris. Kita akan membongkarnya menjadi 17.544 baris menggunakan SQL di tahap Transform!")

except Exception as e:
    print(f"\n[ERROR] Terjadi kegagalan saat memuat data cuaca: {e}")

[STAGE] Memuat Data Weather (100% Pure ELT - Raw JSON)...

[ERROR] Terjadi kegagalan saat memuat data cuaca: (psycopg2.errors.DependentObjectsStillExist) cannot drop table elt_raw_weather because other objects depend on it
DETAIL:  view elt_clean_weather depends on table elt_raw_weather
view elt_std_weather depends on view elt_clean_weather
view elt_fact_311_analytics depends on view elt_std_weather
HINT:  Use DROP ... CASCADE to drop the dependent objects too.

[SQL: 
DROP TABLE elt_raw_weather]
(Background on this error at: https://sqlalche.me/e/20/2j85)


In [22]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

USER = "neondb_owner"
PASSWORD = quote_plus(raw_password)
HOST = "ep-calm-bar-ao8ujkdt-pooler.c-2.ap-southeast-1.aws.neon.tech"
DB = "neondb"
DATABASE_URL = f"postgresql://{USER}:{PASSWORD}@{HOST}/{DB}?sslmode=require"

# KUNCI UTAMA: pool_pre_ping=True agar kebal dari pemutusan sepihak oleh Neon
engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,
    pool_recycle=300
)

print("Koneksi berhasil dibangunkan ulang dan siap digunakan!")

Koneksi berhasil dibangunkan ulang dan siap digunakan!


## PHASE 3: Transform (In-Database Processing)
Ini adalah inti dari arsitektur ELT. Alih-alih menggunakan Pandas, seluruh proses transformasi dijalankan menggunakan SQL murni di dalam Neon DB.

Proses ini mencakup:
1. **Unpacking JSON** menggunakan `json_array_elements_text`.
2. **Cleansing & Standarisasi** (Penanganan nilai null, *outlier clipping* menggunakan IQR, normalisasi Min-Max, dan standarisasi nama Borough menggunakan `INITCAP`).
3. **Data Integration & Feature Engineering** (*Join* tabel 311 dengan tabel cuaca berdasarkan waktu, pembuatan kolom `is_rush_hour`, perhitungan durasi respon, dan kombinasi `rush_hour_bad_weather`).

*Hasil akhir bermuara pada View `elt_fact_311_analytics`.*

In [23]:
from sqlalchemy import text
import time

# ==============================================================================
# TAHAP TRANSFORMASI (MURNI ELT - Dieksekusi di Neon DB)
# ==============================================================================
sql_transform_pipeline = """
-- 1. CLEANUP: Hapus semua view lama agar tidak terjadi konflik
DROP VIEW IF EXISTS elt_fact_311_analytics CASCADE;
DROP VIEW IF EXISTS elt_std_311 CASCADE;
DROP VIEW IF EXISTS elt_clean_311 CASCADE;
DROP VIEW IF EXISTS elt_std_weather CASCADE;
DROP VIEW IF EXISTS elt_clean_weather CASCADE;

-- ==============================================================================
-- 2. PIPELINE CUACA (WEATHER)
-- ==============================================================================
CREATE VIEW elt_clean_weather AS
WITH unpacked_json AS (
    SELECT
        t.time::TIMESTAMP as time, t.temp::NUMERIC as temperature_2m,
        t.precip::NUMERIC as precipitation, t.wind::NUMERIC as wind_speed_10m
    FROM elt_raw_weather,
    ROWS FROM (
        json_array_elements_text(raw_json_data::json->'hourly'->'time'),
        json_array_elements_text(raw_json_data::json->'hourly'->'temperature_2m'),
        json_array_elements_text(raw_json_data::json->'hourly'->'precipitation'),
        json_array_elements_text(raw_json_data::json->'hourly'->'wind_speed_10m')
    ) AS t(time, temp, precip, wind)
),
deduplicated AS (
    SELECT *, ROW_NUMBER() OVER(PARTITION BY time ORDER BY time) as rn
    FROM unpacked_json
),
stats AS (
    SELECT
        percentile_cont(0.5) WITHIN GROUP (ORDER BY precipitation) as precip_med,
        percentile_cont(0.5) WITHIN GROUP (ORDER BY wind_speed_10m) as wind_med,
        percentile_cont(0.25) WITHIN GROUP (ORDER BY precipitation) as precip_q1,
        percentile_cont(0.75) WITHIN GROUP (ORDER BY precipitation) as precip_q3,
        percentile_cont(0.25) WITHIN GROUP (ORDER BY wind_speed_10m) as wind_q1,
        percentile_cont(0.75) WITHIN GROUP (ORDER BY wind_speed_10m) as wind_q3
    FROM deduplicated WHERE rn = 1
),
iqr_calc AS (
    SELECT
        precip_med, wind_med,
        precip_q1 - 1.5 * (precip_q3 - precip_q1) as precip_lower,
        precip_q3 + 1.5 * (precip_q3 - precip_q1) as precip_upper,
        wind_q1 - 1.5 * (wind_q3 - wind_q1) as wind_lower,
        wind_q3 + 1.5 * (wind_q3 - wind_q1) as wind_upper
    FROM stats
)
SELECT
    time, COALESCE(temperature_2m, 0) as temperature_2m,
    CASE
        WHEN COALESCE(precipitation, iqr_calc.precip_med) < iqr_calc.precip_lower THEN iqr_calc.precip_lower
        WHEN COALESCE(precipitation, iqr_calc.precip_med) > iqr_calc.precip_upper THEN iqr_calc.precip_upper
        ELSE COALESCE(precipitation, iqr_calc.precip_med)
    END as precipitation,
    CASE
        WHEN COALESCE(wind_speed_10m, iqr_calc.wind_med) < iqr_calc.wind_lower THEN iqr_calc.wind_lower
        WHEN COALESCE(wind_speed_10m, iqr_calc.wind_med) > iqr_calc.wind_upper THEN iqr_calc.wind_upper
        ELSE COALESCE(wind_speed_10m, iqr_calc.wind_med)
    END as wind_speed_10m
FROM deduplicated CROSS JOIN iqr_calc WHERE rn = 1;

CREATE VIEW elt_std_weather AS
SELECT
    time::TIMESTAMP,
    temperature_2m,
    (temperature_2m - MIN(temperature_2m) OVER()) / NULLIF(MAX(temperature_2m) OVER() - MIN(temperature_2m) OVER(), 0) AS temperature_2m_norm,
    precipitation,
    wind_speed_10m,
    (wind_speed_10m - MIN(wind_speed_10m) OVER()) / NULLIF(MAX(wind_speed_10m) OVER() - MIN(wind_speed_10m) OVER(), 0) AS wind_speed_10m_norm
FROM elt_clean_weather;

-- ==============================================================================
-- 3. PIPELINE NYC 311 (Menggunakan INITCAP untuk Borough)
-- ==============================================================================
CREATE VIEW elt_clean_311 AS
WITH deduplicated AS (
    SELECT *, ROW_NUMBER() OVER(PARTITION BY created_date, complaint_type, incident_address, borough ORDER BY unique_key) as rn
    FROM elt_raw_311 WHERE created_date IS NOT NULL AND complaint_type IS NOT NULL
),
bounds AS (
    SELECT
        percentile_cont(0.25) WITHIN GROUP (ORDER BY latitude::numeric) as lat_q1,
        percentile_cont(0.75) WITHIN GROUP (ORDER BY latitude::numeric) as lat_q3,
        percentile_cont(0.25) WITHIN GROUP (ORDER BY longitude::numeric) as lon_q1,
        percentile_cont(0.75) WITHIN GROUP (ORDER BY longitude::numeric) as lon_q3
    FROM deduplicated WHERE rn = 1
),
iqr_calc AS (
    SELECT
        lat_q1 - 1.5 * (lat_q3 - lat_q1) as lat_lower, lat_q3 + 1.5 * (lat_q3 - lat_q1) as lat_upper,
        lon_q1 - 1.5 * (lon_q3 - lon_q1) as lon_lower, lon_q3 + 1.5 * (lon_q3 - lon_q1) as lon_upper
    FROM bounds
)
SELECT
    unique_key, created_date::TIMESTAMP, closed_date::TIMESTAMP, COALESCE(agency, 'Unknown') as agency,
    CASE
        WHEN UPPER(borough) IN ('MANHATTAN', 'BROOKLYN', 'QUEENS', 'BRONX', 'STATEN ISLAND') THEN INITCAP(borough)
        ELSE 'Unknown'
    END as borough,
    complaint_type, COALESCE(descriptor, 'Unknown') as descriptor, COALESCE(location_type, 'Unknown') as location_type, COALESCE(status, 'Unknown') as status,
    CASE WHEN latitude::numeric < iqr_calc.lat_lower THEN iqr_calc.lat_lower WHEN latitude::numeric > iqr_calc.lat_upper THEN iqr_calc.lat_upper ELSE latitude::numeric END as latitude,
    CASE WHEN longitude::numeric < iqr_calc.lon_lower THEN iqr_calc.lon_lower WHEN longitude::numeric > iqr_calc.lon_upper THEN iqr_calc.lon_upper ELSE longitude::numeric END as longitude
FROM deduplicated CROSS JOIN iqr_calc WHERE rn = 1;

CREATE VIEW elt_std_311 AS
SELECT
    unique_key::BIGINT, created_date, closed_date,
    agency, (DENSE_RANK() OVER (ORDER BY agency)) - 1 AS agency_enc,
    borough, (DENSE_RANK() OVER (ORDER BY borough)) - 1 AS borough_enc,
    complaint_type, (DENSE_RANK() OVER (ORDER BY complaint_type)) - 1 AS complaint_type_enc,
    descriptor, location_type, status,
    latitude, (latitude - MIN(latitude) OVER()) / NULLIF(MAX(latitude) OVER() - MIN(latitude) OVER(), 0) AS latitude_norm,
    longitude, (longitude - MIN(longitude) OVER()) / NULLIF(MAX(longitude) OVER() - MIN(longitude) OVER(), 0) AS longitude_norm
FROM elt_clean_311;

-- ==============================================================================
-- 4. ANALYTICS FACT TABLE (Penggabungan & Feature Engineering)
-- ==============================================================================
CREATE VIEW elt_fact_311_analytics AS
WITH joined_data AS (
    SELECT t1.*, w.temperature_2m_norm, w.precipitation, w.wind_speed_10m_norm
    FROM elt_std_311 t1 LEFT JOIN elt_std_weather w ON DATE_TRUNC('hour', t1.created_date::timestamp) = w.time
),
step1_features AS (
    SELECT *, EXTRACT(HOUR FROM created_date::timestamp) AS hour_of_day, EXTRACT(ISODOW FROM created_date::timestamp) - 1 AS day_of_week, GREATEST(EXTRACT(EPOCH FROM (closed_date::timestamp - created_date::timestamp)) / 3600.0, 0) AS response_time_hours FROM joined_data
),
step2_features AS (
    SELECT *, CASE WHEN day_of_week >= 5 THEN 1 ELSE 0 END AS is_weekend, CASE WHEN hour_of_day IN (7, 8, 9, 16, 17, 18, 19) THEN 1 ELSE 0 END AS is_rush_hour, ROUND((COALESCE(precipitation, 0) + COALESCE(wind_speed_10m_norm, 0))::numeric, 4) AS weather_risk_score, CASE WHEN COALESCE(precipitation, 0) > 0 THEN 1 ELSE 0 END AS is_rainy, CASE WHEN COALESCE(temperature_2m_norm, 0) <= 0.33 THEN 'cold' WHEN COALESCE(temperature_2m_norm, 0) <= 0.66 THEN 'normal' ELSE 'hot' END AS temp_category FROM step1_features
)
SELECT *, CASE WHEN is_rush_hour = 1 AND (is_rainy = 1 OR COALESCE(wind_speed_10m_norm, 0) > 0.5) THEN 1 ELSE 0 END AS rush_hour_bad_weather
FROM step2_features;
"""

print("[STAGE] Memulai Eksekusi Transformasi Data (Pure SQL di Neon DB)...")
start_time = time.time()

try:
    with engine.connect() as conn:
        # Eksekusi seluruh script SQL sekaligus
        conn.execute(text(sql_transform_pipeline))
        conn.commit()

    print(f" -> BERHASIL! View 'elt_fact_311_analytics' dan seluruh dependensinya telah tercipta/diperbarui.")
    print(f"[STAGE] Transformasi Selesai dalam {time.time() - start_time:.2f} detik!\n")

except Exception as e:
    print(f"\n[ERROR] Terjadi kegagalan saat proses transformasi: {e}")

[STAGE] Memulai Eksekusi Transformasi Data (Pure SQL di Neon DB)...
 -> BERHASIL! View 'elt_fact_311_analytics' dan seluruh dependensinya telah tercipta/diperbarui.
[STAGE] Transformasi Selesai dalam 1.56 detik!



## PHASE 4: Data Quality Validation
Melakukan serangkaian pengujian (*Uniqueness, Null Check, Range Check, Referential Integrity*) untuk memastikan tabel fakta akhir sudah benar-benar bersih dan siap dihubungkan ke Dashboard Analitik.

In [24]:
# ============================================================
# Validasi Kualitas Data (Versi ELT/SQL)
# ============================================================
import pandas as pd
import time

print("[VALID-ELT] START | Memulai Validasi Data di PostgreSQL (Pure SQL)...\n")
t_start = time.perf_counter()
passed = 0
failed = 0

# Kita buat fungsi kecil untuk menjalankan query cek anomali
def check_anomaly(query, validasi_name, pass_msg, fail_msg):
    global passed, failed
    result = pd.read_sql(query, engine).iloc[0, 0]
    if result == 0:
        print(f"[{validasi_name}] ✓ PASS | {pass_msg}")
        passed += 1
    else:
        print(f"[{validasi_name}] ✗ FAIL | {fail_msg} ({result} baris anomali ditemukan)")
        failed += 1

# --- VALIDASI 1: Uniqueness Check ---
# Cek apakah ada unique_key yang jumlahnya lebih dari 1
q_unique = """
    SELECT COUNT(*) FROM (
        SELECT unique_key FROM elt_fact_311_analytics
        GROUP BY unique_key HAVING COUNT(*) > 1
    ) AS duplicates
"""
check_anomaly(q_unique, "VALID-1", "Uniqueness: unique_key — 0 duplikat", "Uniqueness: Terdapat duplikat")

# --- VALIDASI 2: Null Check ---
# Cek apakah ada data kosong di kolom wajib
q_null = """
    SELECT COUNT(*) FROM elt_fact_311_analytics
    WHERE unique_key IS NULL OR created_date IS NULL OR complaint_type IS NULL OR borough IS NULL
"""
check_anomaly(q_null, "VALID-2", "Null check mandatory fields: 0 null", "Null ditemukan di mandatory fields")

# --- VALIDASI 3: Range Check ---
# Cek apakah latitude/longitude normalisasi keluar dari rentang 0-1
q_range = """
    SELECT COUNT(*) FROM elt_fact_311_analytics
    WHERE latitude_norm < 0 OR latitude_norm > 1 OR longitude_norm < 0 OR longitude_norm > 1
"""
check_anomaly(q_range, "VALID-3", "Range check lat/lon (normalisasi [0,1]): semua valid", "Range lat/lon invalid")

# --- VALIDASI 4: Datatype Consistency ---
# Dalam database relasional seperti PostgreSQL, tipe data dikunci ketat oleh sistem.
# Berbeda dengan Pandas, mustahil ada tipe data yang tercampur di satu kolom SQL.
print("[VALID-4] ✓ PASS | Datatype consistency: Dijamin 100% konsisten oleh mesin PostgreSQL")
passed += 1

# --- VALIDASI 5: Referential Integrity ---
# Cek apakah ada Borough di luar daftar yang diizinkan
q_ref = """
    SELECT COUNT(*) FROM elt_fact_311_analytics
    WHERE UPPER(borough) NOT IN ('MANHATTAN', 'BROOKLYN', 'QUEENS', 'BRONX', 'STATEN ISLAND', 'UNKNOWN')
"""
check_anomaly(q_ref, "VALID-5", "Referential integrity: semua nilai borough valid", "Nilai borough tidak valid ditemukan")

# --- VALIDASI 6: Distribusi Data ---
# Cek apakah ada durasi response yang bernilai negatif
q_dist = """
    SELECT COUNT(*) FROM elt_fact_311_analytics
    WHERE response_time_hours < 0
"""
check_anomaly(q_dist, "VALID-6", "Distribusi: response_time_hours — 0 nilai negatif", "Nilai negatif di response_time_hours")

t_elapsed = time.perf_counter() - t_start
print(f"\n[VALID-ELT] DONE | PASS: {passed}/6 | FAIL: {failed}/6 | Waktu: {t_elapsed:.2f}s")
print("🚀 Data Analytics Fact Table di Neon DB dinyatakan BERSIH dan SIAP ditarik ke Power BI!")

[VALID-ELT] START | Memulai Validasi Data di PostgreSQL (Pure SQL)...

[VALID-1] ✓ PASS | Uniqueness: unique_key — 0 duplikat
[VALID-2] ✓ PASS | Null check mandatory fields: 0 null
[VALID-3] ✓ PASS | Range check lat/lon (normalisasi [0,1]): semua valid
[VALID-4] ✓ PASS | Datatype consistency: Dijamin 100% konsisten oleh mesin PostgreSQL
[VALID-5] ✓ PASS | Referential integrity: semua nilai borough valid
[VALID-6] ✓ PASS | Distribusi: response_time_hours — 0 nilai negatif

[VALID-ELT] DONE | PASS: 6/6 | FAIL: 0/6 | Waktu: 20.70s
🚀 Data Analytics Fact Table di Neon DB dinyatakan BERSIH dan SIAP ditarik ke Power BI!


## PHASE 5: Analitik & Ekstraksi Insight (Data Mart)
Fase terakhir ini membuktikan bahwa tabel fakta (`elt_fact_311_analytics`) yang telah dibangun di *Data Warehouse* sudah siap untuk dihubungkan ke *Dashboard BI* atau dianalisis lebih lanjut.

Di sini, kita menggunakan Pandas hanya sebagai media eksekutor (penampil hasil). Beban komputasi yang berat untuk mengagregasi ratusan ribu baris data tetap dieksekusi murni oleh *engine* PostgreSQL di Neon DB.

Berikut adalah eksekusi 8 *query* analitik untuk menjawab berbagai pertanyaan bisnis terkait operasional layanan NYC 311 dan korelasinya dengan cuaca:

In [25]:
import pandas as pd

print("📊 8 QUERY SQL ANALITIK (Berdasarkan Tabel Fact ELT)\n" + "="*50)

queries = {
    "1. Top 5 Wilayah (Borough) dengan Komplain Terbanyak": """
        SELECT borough, COUNT(unique_key) as total_requests
        FROM elt_fact_311_analytics
        GROUP BY borough ORDER BY total_requests DESC LIMIT 5;
    """,

    "2. Top 10 Jenis Komplain Warga NYC": """
        SELECT complaint_type, COUNT(unique_key) as total_requests
        FROM elt_fact_311_analytics
        GROUP BY complaint_type ORDER BY total_requests DESC LIMIT 10;
    """,

    "3. Tren Komplain Berdasarkan Jam Kejadian": """
        SELECT hour_of_day, COUNT(unique_key) as total_requests
        FROM elt_fact_311_analytics
        GROUP BY hour_of_day ORDER BY hour_of_day ASC;
    """,

    "4. Perbandingan Komplain: Hari Kerja vs Akhir Pekan": """
        SELECT CASE WHEN is_weekend = 1 THEN 'Weekend' ELSE 'Weekday' END as day_type,
               COUNT(unique_key) as total_requests
        FROM elt_fact_311_analytics
        GROUP BY day_type;
    """,

    "5. Rata-rata Waktu Penanganan (Response Time) per Jenis Komplain": """
        SELECT complaint_type, ROUND(AVG(response_time_hours)::numeric, 2) as avg_response_hours
        FROM elt_fact_311_analytics
        WHERE response_time_hours IS NOT NULL
        GROUP BY complaint_type ORDER BY avg_response_hours DESC LIMIT 5;
    """,

    "6. Korelasi Cuaca Buruk di Jam Sibuk (Rush Hour) Terhadap Volume Komplain": """
        SELECT CASE WHEN rush_hour_bad_weather = 1 THEN 'Jam Sibuk & Cuaca Buruk'
                    ELSE 'Kondisi Normal' END as kondisi,
               COUNT(unique_key) as total_requests
        FROM elt_fact_311_analytics
        GROUP BY kondisi;
    """,

    "7. Rata-rata Risiko Cuaca (Weather Risk Score) di Setiap Borough": """
        SELECT borough, ROUND(AVG(weather_risk_score)::numeric, 3) as avg_weather_risk
        FROM elt_fact_311_analytics
        GROUP BY borough ORDER BY avg_weather_risk DESC;
    """,

    "8. Distribusi Keluhan 'Street Condition' Berdasarkan Kategori Suhu Udara": """
        SELECT
            CASE
                WHEN temperature_2m_norm < 0.3 THEN 'Dingin'
                WHEN temperature_2m_norm > 0.7 THEN 'Panas'
                ELSE 'Normal'
            END as kategori_suhu,
            COUNT(unique_key) as total_street_condition_complaints
        FROM elt_fact_311_analytics
        WHERE complaint_type ILIKE '%%Street Condition%%'
        GROUP BY kategori_suhu ORDER BY total_street_condition_complaints DESC;
    """
}

# Mengeksekusi dan menampilkan hasil dari 8 query
for title, query in queries.items():
    print(f"\n📌 {title}")
    display(pd.read_sql(query, engine))

📊 8 QUERY SQL ANALITIK (Berdasarkan Tabel Fact ELT)

📌 1. Top 5 Wilayah (Borough) dengan Komplain Terbanyak


,borough,total_requests
0,Brooklyn,176319
1,Queens,160505
2,Bronx,77818
3,Manhattan,60605
4,Staten Island,18038



📌 2. Top 10 Jenis Komplain Warga NYC


,complaint_type,total_requests
0,Illegal Parking,290267
1,Blocked Driveway,98505
2,Street Condition,42541
3,Traffic Signal Condition,22352
4,Street Light Condition,19632
5,Sidewalk Condition,13970
6,Curb Condition,3900
7,Highway Condition,2109
8,Bus Stop Shelter Complaint,647



📌 3. Tren Komplain Berdasarkan Jam Kejadian


...

,kondisi,total_requests
0,Jam Sibuk & Cuaca Buruk,77072
1,Kondisi Normal,416851



📌 7. Rata-rata Risiko Cuaca (Weather Risk Score) di Setiap Borough


,borough,avg_weather_risk
0,Unknown,0.546
1,Manhattan,0.471
2,Staten Island,0.466
3,Brooklyn,0.462
4,Queens,0.458
5,Bronx,0.451



📌 8. Distribusi Keluhan 'Street Condition' Berdasarkan Kategori Suhu Udara


,kategori_suhu,total_street_condition_complaints
0,Normal,28588
1,Panas,12422
2,Dingin,1531
